# Evaluation: Dear LLM, Can You Explain my MongoDB Database?

Computes the metrics of the Notion *Evaluation* page (Fields F1, Relationships F1, Detection F1, Classification accuracy, End-to-end F1, confusion matrix), in **strict** and **normalized** mode, aggregated by pooling.

How to use: run the cells in order. To evaluate another model, change only `MODEL` and `RUN_DIRS` in cell 2.

Output: `Repository/results/<MODEL>/` on Drive.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Configuration

In [16]:
import os

ROOT = "/content/drive/MyDrive/Applied Data Science Project/Repository"

PROMPT_NAME = "zero_shot"                 # same name used in the generation notebook
MODEL = "claude-opus-5.5"                        # file tag
RUN_DIRS = [f"{ROOT}/predictions/{MODEL}_{PROMPT_NAME}/Run {i}" for i in (1, 2, 3)]
THRESHOLD = 0.51

TEST_CASES = f"{ROOT}/Test Cases"
GT_DIR = "/content/GT"
SRC_DIR = f"{ROOT}/src"
OUT_DIR = f"{ROOT}/results/{MODEL}_{PROMPT_NAME}"   # separate folder, the baseline results stay untouched

os.makedirs(OUT_DIR, exist_ok=True)
os.chdir(SRC_DIR)
print("scripts:", [f for f in os.listdir(SRC_DIR) if f in ("normalize.py", "evaluate.py")])
for d in RUN_DIRS:
    print(d, "->", "ok" if os.path.isdir(d) else "MISSING")

scripts: ['evaluate.py', 'normalize.py']
/content/drive/MyDrive/Applied Data Science Project/Repository/predictions/claude-opus-5.5_zero_shot/Run 1 -> ok
/content/drive/MyDrive/Applied Data Science Project/Repository/predictions/claude-opus-5.5_zero_shot/Run 2 -> ok
/content/drive/MyDrive/Applied Data Science Project/Repository/predictions/claude-opus-5.5_zero_shot/Run 3 -> ok


## 3. Collect the GT files in one folder
On Drive the GT files are split across Simple / Medium / Hard folders.

In [17]:
import glob, shutil

os.makedirs(GT_DIR, exist_ok=True)
gt_files = sorted(glob.glob(f"{TEST_CASES}/**/*_GT.json", recursive=True))
for f in gt_files:
    shutil.copy(f, GT_DIR)
print(len(gt_files), "GT files:", sorted(os.listdir(GT_DIR)))

12 GT files: ['easy1_GT.json', 'easy2_GT.json', 'easy3_GT.json', 'easy4_GT.json', 'hard10_GT.json', 'hard11_GT.json', 'hard12_GT.json', 'hard9_GT.json', 'medium5_GT.json', 'medium6_GT.json', 'medium7_GT.json', 'medium8_GT.json']


## 4. Run the evaluation

In [18]:
run_args = " ".join(f'"{d}"' for d in RUN_DIRS)
!python evaluate.py --gt-dir "{GT_DIR}" --run-dirs {run_args} --model {MODEL} --threshold {THRESHOLD} --out-dir "{OUT_DIR}"
print("\nOutput files:", sorted(os.listdir(OUT_DIR)))

{'mode': 'normalized', 'group': 'easy', 'n_db': 4, 'n_runs': 12, 'fields_f1': 1.0, 'relationships_f1': 1.0, 'detection_f1': 1.0, 'classification_acc': 1.0, 'e2e_f1': 1.0}
{'mode': 'normalized', 'group': 'medium', 'n_db': 4, 'n_runs': 12, 'fields_f1': 1.0, 'relationships_f1': 0.879, 'detection_f1': 1.0, 'classification_acc': 0.95, 'e2e_f1': 0.95}
{'mode': 'normalized', 'group': 'hard', 'n_db': 4, 'n_runs': 12, 'fields_f1': 1.0, 'relationships_f1': 0.853, 'detection_f1': 0.983, 'classification_acc': 1.0, 'e2e_f1': 0.983}
{'mode': 'normalized', 'group': 'global', 'n_db': 12, 'n_runs': 36, 'fields_f1': 1.0, 'relationships_f1': 0.881, 'detection_f1': 0.991, 'classification_acc': 0.984, 'e2e_f1': 0.975}

Output files: ['confusion_normalized_claude-opus-5.5.csv', 'issues_claude-opus-5.5.json', 'per_db_claude-opus-5.5.csv', 'per_run_claude-opus-5.5.csv', 'summary_claude-opus-5.5.csv']


## 5. Results
All values are pooled: TP, FP and FN summed over runs and databases, metrics computed once on the totals.

### Summary (per level and global)

In [19]:
import pandas as pd
summary = pd.read_csv(f"{OUT_DIR}/summary_{MODEL}.csv")
cols = ["mode", "group", "n_db", "n_runs", "fields_f1", "relationships_f1", "detection_f1", "classification_acc", "e2e_f1"]
summary[cols]

,mode,group,n_db,n_runs,fields_f1,relationships_f1,detection_f1,classification_acc,e2e_f1
0,normalized,easy,4,12,1.0,1.000,1.000,1.000,1.000
1,normalized,medium,4,12,1.0,0.879,1.000,0.950,0.950
2,normalized,hard,4,12,1.0,0.853,0.983,1.000,0.983
3,normalized,global,12,36,1.0,0.881,0.991,0.984,0.975


### Per database, pooled over the runs

In [20]:
per_db = pd.read_csv(f"{OUT_DIR}/per_db_{MODEL}.csv")
cols = ["mode", "level", "db", "n_runs", "fields_f1", "relationships_f1", "detection_f1", "classification_acc", "e2e_f1"]
per_db[per_db["mode"] == "normalized"][cols]

,mode,level,db,n_runs,fields_f1,relationships_f1,detection_f1,classification_acc,e2e_f1
0,normalized,easy,easy1,3,1.0,1.000,1.000,1.000,1.000
1,normalized,easy,easy2,3,1.0,1.000,1.000,1.000,1.000
2,normalized,easy,easy3,3,1.0,1.000,1.000,1.000,1.000
3,normalized,easy,easy4,3,1.0,1.000,1.000,1.000,1.000
4,normalized,hard,hard10,3,1.0,0.818,0.968,1.000,0.968
5,normalized,hard,hard11,3,1.0,0.857,0.989,1.000,0.989
6,normalized,hard,hard12,3,1.0,0.900,0.992,1.000,0.992
7,normalized,hard,hard9,3,1.0,0.824,0.980,1.000,0.980
8,normalized,medium,medium5,3,1.0,0.800,1.000,1.000,1.000
9,normalized,medium,medium6,3,1.0,1.000,1.000,0.833,0.833


### Confusion matrix (normalized mode, summed over all databases and runs)
Rows = GT pattern, columns = predicted pattern.

In [14]:
pd.read_csv(f"{OUT_DIR}/confusion_normalized_{MODEL}.csv", index_col=0)

,Embedding,Referencing,ExtendedReference,Subset,Array,Bucket,Conflict
GT \ Pred,,,,,,,
Embedding,64,0,0,0,22,0,0
Referencing,0,61,0,0,3,0,0
ExtendedReference,0,0,27,0,0,0,0
Subset,0,0,1,17,0,0,0
Array,0,0,0,0,54,0,0
Bucket,0,0,0,0,0,18,0


### Issues log (what normalize.py noticed in the model outputs)

In [21]:
import json
issues = json.load(open(f"{OUT_DIR}/issues_{MODEL}.json"))
for k, v in issues.items():
    print(k)
    for line in v:
        print("   ", line)